In [1]:
from pathlib import Path

import pandas as pd

In [2]:
dataset_path = Path("../dataset")
accounts_path = dataset_path / "HI-Small_accounts.csv"
transactions_path = dataset_path / "HI-Small_Trans.csv"
print(f"{accounts_path=}")
print(f"{transactions_path=}")

accounts_path=PosixPath('../dataset/HI-Small_accounts.csv')
transactions_path=PosixPath('../dataset/HI-Small_Trans.csv')


# Accounts

In [3]:
accounts = pd.read_csv(accounts_path)
print(f"{accounts.shape=}")
accounts.head()

accounts.shape=(518581, 5)


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [4]:
accounts['Bank Name'] = accounts['Bank Name'].str.split(' #').str[0]
accounts['Entity Name'] = accounts['Entity Name'].str.split(' #').str[0]

# Transactions

In [5]:
transactions = pd.read_csv(transactions_path)
transactions = transactions.rename(columns={"Account": "From Account", "Account.1": "To Account"})
transactions['Timestamp'] = pd.to_datetime(transactions['Timestamp'])
print(f"{transactions.shape=}")
transactions.head()

transactions.shape=(5078345, 11)


,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


# Data Modelling

In [6]:
import math

import joblib
import networkx as nx
import torch
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.utils import to_networkx

/Users/darrellcr/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vertex (Account)
features:
- bank id
- bank name
- account number
- entity name

Edge (Transaction)
features:
- Amount Received
- Amount Paid
- Receiving Currency
- Payment Currency
- Payment Format

## Vertex

In [7]:
accounts_iterator = accounts.loc[:, ['Bank ID', 'Account Number']].itertuples()

account_to_idx = {(bank, account): i for i, (_, bank, account) in enumerate(accounts_iterator)}

In [8]:
transformer = ColumnTransformer([
    ('categorical_features', OneHotEncoder(sparse_output=False, drop='first'), ['Entity Name'])
], remainder='drop')


vertices = torch.tensor(transformer.fit_transform(accounts), dtype=torch.float)

joblib.dump(transformer, './preprocessors/vertex_preprocessor.joblib')

['./preprocessors/vertex_preprocessor.joblib']

## Edge

### Connections

In [9]:
transactions.head()

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0


In [10]:
source_keys = list(zip(transactions['From Bank'], transactions['From Account']))
destination_keys = list(zip(transactions['To Bank'], transactions['To Account']))

transactions['source'] = pd.Series(source_keys).map(account_to_idx)
transactions['destination'] = pd.Series(destination_keys).map(account_to_idx)
transactions.head()

,Timestamp,From Bank,From Account,To Bank,To Account,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,source,destination
0,2022-09-01 00:20:00,10,8000EBD30,10,8000EBD30,3697.34,US Dollar,3697.34,US Dollar,Reinvestment,0,435512,435512
1,2022-09-01 00:20:00,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0,65242,474699
2,2022-09-01 00:00:00,3209,8000F4670,3209,8000F4670,14675.57,US Dollar,14675.57,US Dollar,Reinvestment,0,65597,65597
3,2022-09-01 00:02:00,12,8000F5030,12,8000F5030,2806.97,US Dollar,2806.97,US Dollar,Reinvestment,0,475408,475408
4,2022-09-01 00:06:00,10,8000F5200,10,8000F5200,36682.97,US Dollar,36682.97,US Dollar,Reinvestment,0,475619,475619


In [11]:
transactions[['source', 'destination']].values.T

array([[435512,  65242,  65597, ..., 460201, 430481, 497197],
       [435512, 474699,  65597, ..., 405368, 405368, 405368]],
      shape=(2, 5078345))

In [12]:
edge_index = torch.tensor(transactions[['source', 'destination']].values.T, dtype=torch.long)

### Features

In [13]:
categorical_features = ['Payment Format', 'Receiving Currency', 'Payment Currency']
numerical_features = ['Amount Received', 'Amount Paid']

transformer = ColumnTransformer([
    ('categorical', OneHotEncoder(sparse_output=False), categorical_features),
    ('numerical', RobustScaler(), numerical_features),
], remainder='passthrough', verbose_feature_names_out=False)

transformer.set_output(transform='pandas')

preprocessed_transactions = transformer.fit_transform(transactions)
to_remove = ['From Bank', 'From Account', 'To Account', 'To Bank', 'Is Laundering', 'Timestamp']
preprocessed_transactions = preprocessed_transactions.drop(columns=to_remove)
preprocessed_transactions.columns

Index(['Payment Format_ACH', 'Payment Format_Bitcoin', 'Payment Format_Cash',
       'Payment Format_Cheque', 'Payment Format_Credit Card',
       'Payment Format_Reinvestment', 'Payment Format_Wire',
       'Receiving Currency_Australian Dollar', 'Receiving Currency_Bitcoin',
       'Receiving Currency_Brazil Real', 'Receiving Currency_Canadian Dollar',
       'Receiving Currency_Euro', 'Receiving Currency_Mexican Peso',
       'Receiving Currency_Ruble', 'Receiving Currency_Rupee',
       'Receiving Currency_Saudi Riyal', 'Receiving Currency_Shekel',
       'Receiving Currency_Swiss Franc', 'Receiving Currency_UK Pound',
       'Receiving Currency_US Dollar', 'Receiving Currency_Yen',
       'Receiving Currency_Yuan', 'Payment Currency_Australian Dollar',
       'Payment Currency_Bitcoin', 'Payment Currency_Brazil Real',
       'Payment Currency_Canadian Dollar', 'Payment Currency_Euro',
       'Payment Currency_Mexican Peso', 'Payment Currency_Ruble',
       'Payment Currency_Rupee'

In [14]:
joblib.dump(transformer, './preprocessors/edge_preprocessor.joblib')

['./preprocessors/edge_preprocessor.joblib']

In [14]:
preprocessed_transactions[numerical_features].head()

,Amount Received,Amount Paid
0,0.187976,0.188453
1,-0.116009,-0.116774
2,1.090575,1.094744
3,0.114772,0.114950
4,2.899963,2.911532


In [15]:
preprocessed_transactions.loc[:, ~preprocessed_transactions.columns.isin(numerical_features)]

,Payment Format_ACH,Payment Format_Bitcoin,Payment Format_Cash,Payment Format_Cheque,Payment Format_Credit Card,Payment Format_Reinvestment,Payment Format_Wire,Receiving Currency_Australian Dollar,Receiving Currency_Bitcoin,Receiving Currency_Brazil Real,...,Payment Currency_Rupee,Payment Currency_Saudi Riyal,Payment Currency_Shekel,Payment Currency_Swiss Franc,Payment Currency_UK Pound,Payment Currency_US Dollar,Payment Currency_Yen,Payment Currency_Yuan,source,destination
0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,435512,435512
1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,65242,474699
2,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,65597,65597
3,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,475408,475408
4,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,475619,475619
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5078340,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,228536,405368
5078341,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,162963,405368
5078342,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,460201,405368
5078343,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,430481,405368


In [16]:
preprocessed_transactions.columns.difference(numerical_features, sort=False)

Index(['Payment Format_ACH', 'Payment Format_Bitcoin', 'Payment Format_Cash',
       'Payment Format_Cheque', 'Payment Format_Credit Card',
       'Payment Format_Reinvestment', 'Payment Format_Wire',
       'Receiving Currency_Australian Dollar', 'Receiving Currency_Bitcoin',
       'Receiving Currency_Brazil Real', 'Receiving Currency_Canadian Dollar',
       'Receiving Currency_Euro', 'Receiving Currency_Mexican Peso',
       'Receiving Currency_Ruble', 'Receiving Currency_Rupee',
       'Receiving Currency_Saudi Riyal', 'Receiving Currency_Shekel',
       'Receiving Currency_Swiss Franc', 'Receiving Currency_UK Pound',
       'Receiving Currency_US Dollar', 'Receiving Currency_Yen',
       'Receiving Currency_Yuan', 'Payment Currency_Australian Dollar',
       'Payment Currency_Bitcoin', 'Payment Currency_Brazil Real',
       'Payment Currency_Canadian Dollar', 'Payment Currency_Euro',
       'Payment Currency_Mexican Peso', 'Payment Currency_Ruble',
       'Payment Currency_Rupee'

In [17]:
edge_features = torch.tensor(preprocessed_transactions.values, dtype=torch.float)
print(edge_features.shape)
print(edge_features[:2])


torch.Size([5078345, 41])
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  1.8798e-01,  1.8845e-01,  4.3551e+05,
          4.3551e+05],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0

### Label

In [18]:
edge_label = transactions['Is Laundering'].values
edge_label

array([0, 0, 0, ..., 0, 0, 0], shape=(5078345,))

In [19]:
from datetime import datetime
import random
from time import perf_counter
from typing import Literal

import numpy as np
import torch
from sklearn.metrics import classification_report
from tqdm import tqdm
from torch.nn import Linear
from torch.optim import AdamW
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
from torch_geometric.data import TemporalData
from torch_geometric.loader import TemporalDataLoader
from torch_geometric.nn import TGNMemory, TransformerConv
from torch_geometric.nn.models.tgn import (
    LastAggregator,
    IdentityMessage,
    LastNeighborLoader,
)
from torch.utils.tensorboard import SummaryWriter

In [40]:
device = torch.device('mps')
unix_timestamp = transactions['Timestamp'].astype('int64') // 10**9
data = TemporalData(
    src=edge_index[0],
    dst=edge_index[1],
    t=torch.tensor(unix_timestamp.values, dtype=torch.float),
    msg=edge_features,
    y=torch.tensor(transactions['Is Laundering'].values, dtype=torch.float)
).to(device)

In [41]:
train_data, val_data, test_data = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)

# Keep natural class labels from the dataset; do not inject synthetic negatives.
train_loader = TemporalDataLoader(train_data, batch_size=512)
val_loader = TemporalDataLoader(val_data, batch_size=512)
test_loader = TemporalDataLoader(test_data, batch_size=512)

neighbor_loader = LastNeighborLoader(data.num_nodes, size=15, device=device)

# Model

In [42]:
def set_seed(seed) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.mps.manual_seed(seed)
    torch.use_deterministic_algorithms(True)

In [43]:
class GraphAttentionEmbedding(torch.nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, time_enc):
        super().__init__()
        self.time_enc = time_enc
        edge_dim = msg_dim + time_enc.out_channels
        self.conv = TransformerConv(in_channels, out_channels // 2, heads=2,
                                    dropout=0.1, edge_dim=edge_dim)

    def forward(self, x, last_update, edge_index, t, msg):
        rel_t = last_update[edge_index[0]] - t
        rel_t_enc = self.time_enc(rel_t.to(x.dtype))
        edge_attr = torch.cat([rel_t_enc, msg], dim=-1)
        return self.conv(x, edge_index, edge_attr)
    

# Transformed LinkPredictor into EdgeClassifier
class EdgeClassifier(torch.nn.Module):
    def __init__(self, in_channels, edge_dim):
        super().__init__()
        # Concatenate src_node, dst_node, and raw edge_attr
        self.lin = Linear(in_channels * 2 + edge_dim, in_channels)
        self.lin_final = Linear(in_channels, 1)

    def forward(self, z_src, z_dst, edge_attr):
        h = torch.cat([z_src, z_dst, edge_attr], dim=-1)
        h = self.lin(h).relu()
        return self.lin_final(h)

In [8]:
class TGNTrainer:
    def __init__(
        self,
        memory: TGNMemory,
        gnn: torch.nn.Module,
        edge_classifier: torch.nn.Module,
        data: TemporalData,
        neighbor_loader,
        train_loader,
        val_loader,
        criterion,
        Optimizer: type[torch.optim.Optimizer],
        device: Literal['cpu', 'mps'] = "cpu",
    ) -> None:
        self.device = device

        self.memory = memory.to(self.device)
        self.gnn = gnn.to(self.device)
        self.edge_classifier = edge_classifier.to(self.device)

        self.data = data
        self.neighbor_loader = neighbor_loader
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.optimizer = Optimizer(
            set(self.memory.parameters()) | set(self.gnn.parameters()) | set(self.edge_classifier.parameters()),
            lr=0.0001,
        )
        self.criterion = criterion.to(self.device)

        self.epochs_run = 0
        self.assoc = torch.empty(data.num_nodes, dtype=torch.long, device=self.device)

    @staticmethod
    def _logits_to_pred(logits: torch.Tensor) -> torch.Tensor:
        # BCEWithLogitsLoss uses raw logits; threshold at 0 == sigmoid 0.5.
        return (logits.squeeze(-1) > 0).to(torch.int64)

    def _run_batch(self, batch):
        batch = batch.to(self.device)
        self.optimizer.zero_grad()

        n_id, edge_index, e_id = self.neighbor_loader(batch.n_id)
        self.assoc[n_id] = torch.arange(n_id.size(0), device=self.device)

        z, last_update = self.memory(n_id)
        z = self.gnn(
            z,
            last_update,
            edge_index,
            self.data.t[e_id].to(self.device),
            self.data.msg[e_id].to(self.device)
        )

        out = self.edge_classifier(z[self.assoc[batch.src]], z[self.assoc[batch.dst]], batch.msg)
        
        loss = self.criterion(out.squeeze(), batch.y.float())
        self.memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        self.neighbor_loader.insert(batch.src, batch.dst)

        loss.backward()
        self.optimizer.step()
        self.memory.detach()

        return loss.item() * batch.num_events, batch.num_events, self._logits_to_pred(out).detach().cpu(), batch.y.to(torch.int64).detach().cpu()

    def _run_epoch(self, epoch: int):
        progress_bar = tqdm(enumerate(self.train_loader), total=len(self.train_loader))
        loss = 0.0
        num_events = 0
        epoch_pred = []
        epoch_true = []

        for i, batch in progress_bar:
            loss_val, batch_num_events, batch_pred, batch_true = self._run_batch(batch)
            loss += loss_val
            num_events += batch_num_events

            epoch_pred.append(batch_pred)
            epoch_true.append(batch_true)
            progress_bar.set_postfix({"loss": f"{(loss/num_events):.4f}"})


        avg_loss = loss / num_events
        train_pred = torch.cat(epoch_pred).numpy()
        train_true = torch.cat(epoch_true).numpy()
        # print(f"{train_pred=}")
        # print(f"{train_true=}")

        return avg_loss, train_true, train_pred

    @torch.no_grad()
    def _evaluate(self, epoch):
        loss = 0.0
        epoch_pred = []
        epoch_true = []

        for i, batch in enumerate(self.val_loader):
            batch = batch.to(self.device)
            pred = self.model(batch.x, batch.edge_index, batch.edge_label_index, batch.edge_attr)
            batch_loss = self.criterion(pred, batch.edge_label.unsqueeze(-1))

            epoch_pred.append(self._logits_to_pred(pred).detach().cpu())
            epoch_true.append(batch.edge_label.to(torch.int64).detach().cpu())
            loss += batch_loss.item()

        avg_loss = loss / len(self.val_loader)
        val_pred = torch.cat(epoch_pred).numpy()
        val_true = torch.cat(epoch_true).numpy()

        return avg_loss, val_true, val_pred

    @staticmethod
    def _log_mps_memory(tag: str) -> None:
        if not torch.backends.mps.is_available():
            return
        torch.mps.synchronize()
        current_gb = torch.mps.current_allocated_memory() / (1024 ** 3)
        driver_gb = torch.mps.driver_allocated_memory() / (1024 ** 3)
        print(f"[{tag}] MPS current={current_gb:.2f} GB | driver={driver_gb:.2f} GB")

    def eval_mode(self):
        self.memory.eval()
        self.gnn.eval()
        self.edge_classifier.eval()
    
    def train_mode(self):
        self.memory.train()
        self.gnn.train()
        self.edge_classifier.train()

    def _reset_temporal_state(self) -> None:
        # Ensure edge ids from neighbor sampling start from the current split each epoch.
        self.memory.reset_state()
        self.neighbor_loader.reset_state()

    def train(self, max_epochs: int) -> None:
        run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
        with SummaryWriter(f"logs/tensorboard/{run_id}") as writer:
            for epoch in range(self.epochs_run, max_epochs):
                self._reset_temporal_state()
                start_time = perf_counter()
                self.train_mode()
                avg_loss, train_true, train_pred = self._run_epoch(epoch)
                time_elapsed = perf_counter() - start_time

                writer.add_scalar("Loss/train", avg_loss, epoch)
                print(f"Epoch: {epoch} | Time: {time_elapsed:.2f}s | Loss: {avg_loss:.3f}")
                print(classification_report(train_true, train_pred, digits=5))

                # self.eval_mode()
                # val_loss, val_true, val_pred = self._evaluate(epoch)
                # print(f"Epoch: {epoch} | Validation Loss: {val_loss:.3f}")
                # # print(classification_report(val_true, val_pred, digits=5))
                # writer.add_scalar("Loss/validation", val_loss, epoch)

                # self._log_mps_memory(f"epoch={epoch}")
                # if torch.backends.mps.is_available():
                #     torch.mps.empty_cache()

                self.epochs_run += 1

NameError: name 'TGNMemory' is not defined

In [9]:
set_seed(124)

embedding_dim = memory_dim = time_dim = 64
memory = TGNMemory(
    data.num_nodes,
    data.msg.size(-1),
    memory_dim,
    time_dim,
    message_module=IdentityMessage(data.msg.size(-1), memory_dim, time_dim),
    aggregator_module=LastAggregator(),
)

gnn = GraphAttentionEmbedding(
    in_channels=memory_dim,
    out_channels=embedding_dim,
    msg_dim=data.msg.size(-1),
    time_enc=memory.time_enc,
)

edge_classifier = EdgeClassifier(in_channels=embedding_dim, edge_dim=data.msg.size(-1))

Optimizer = AdamW
value_counts = transactions['Is Laundering'].value_counts().values
weight = torch.tensor([500], dtype=torch.float)
criterion = torch.nn.BCEWithLogitsLoss(pos_weight=weight)

trainer = TGNTrainer(
    memory=memory,
    gnn=gnn,
    edge_classifier=edge_classifier,
    data=data,
    neighbor_loader=neighbor_loader,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    Optimizer=Optimizer,
    device=device
)
trainer.train(max_epochs=5)

NameError: name 'set_seed' is not defined

In [10]:
---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
Cell In[27], line 39
     35     criterion=criterion,
     36     Optimizer=Optimizer,
     37     device=device
     38 )
---> 39 trainer.train(max_epochs=5)

Cell In[26], line 144, in TGNTrainer.train(self, max_epochs)
    140             for epoch in range(self.epochs_run, max_epochs):
    141                 self._reset_temporal_state()
    142                 start_time = perf_counter()
    143                 self.train_mode()
--> 144                 avg_loss, train_true, train_pred = self._run_epoch(epoch)
    145                 time_elapsed = perf_counter() - start_time
    146 
    147                 writer.add_scalar("Loss/train", avg_loss, epoch)

Cell In[26], line 75, in TGNTrainer._run_epoch(self, epoch)
     71         epoch_pred = []
     72         epoch_true = []
     73 
     74         for i, batch in progress_bar:
---> 75             loss_val, batch_num_events, batch_pred, batch_true = self._run_batch(batch)
     76             loss += loss_val
     77             num_events += batch_num_events
     78 

Cell In[26], line 46, in TGNTrainer._run_batch(self, batch)
     42 
     43         n_id, edge_index, e_id = self.neighbor_loader(batch.n_id)
     44         self.assoc[n_id] = torch.arange(n_id.size(0), device=self.device)
     45 
---> 46         z, last_update = self.memory(n_id)
     47         z = self.gnn(
     48             z,
     49             last_update,

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch/nn/modules/module.py:1779, in Module._wrapped_call_impl(self, *args, **kwargs)
   1777     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1778 else:
-> 1779     return self._call_impl(*args, **kwargs)

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch/nn/modules/module.py:1790, in Module._call_impl(self, *args, **kwargs)
   1785 # If we don't have any hooks, we want to skip the rest of the logic in
   1786 # this function, and just call forward.
   1787 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1788         or _global_backward_pre_hooks or _global_backward_hooks
   1789         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1790     return forward_call(*args, **kwargs)
   1792 result = None
   1793 called_always_called_hooks = set()

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch_geometric/nn/models/tgn.py:96, in TGNMemory.forward(self, n_id)
     92 """Returns, for all nodes :obj:`n_id`, their current memory and their
     93 last updated timestamp.
     94 """
     95 if self.training:
---> 96     memory, last_update = self._get_updated_memory(n_id)
     97 else:
     98     memory, last_update = self.memory[n_id], self.last_update[n_id]

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch_geometric/nn/models/tgn.py:145, in TGNMemory._get_updated_memory(self, n_id)
    143 msg = torch.cat([msg_s, msg_d], dim=0)
    144 t = torch.cat([t_s, t_d], dim=0)
--> 145 aggr = self.aggr_module(msg, self._assoc[idx], t, n_id.size(0))
    147 # Get local copy of updated memory.
    148 memory = self.gru(aggr, self.memory[n_id])

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch/nn/modules/module.py:1779, in Module._wrapped_call_impl(self, *args, **kwargs)
   1777     return self._compiled_call_impl(*args, **kwargs)  # type: ignore[misc]
   1778 else:
-> 1779     return self._call_impl(*args, **kwargs)

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch/nn/modules/module.py:1790, in Module._call_impl(self, *args, **kwargs)
   1785 # If we don't have any hooks, we want to skip the rest of the logic in
   1786 # this function, and just call forward.
   1787 if not (self._backward_hooks or self._backward_pre_hooks or self._forward_hooks or self._forward_pre_hooks
   1788         or _global_backward_pre_hooks or _global_backward_hooks
   1789         or _global_forward_hooks or _global_forward_pre_hooks):
-> 1790     return forward_call(*args, **kwargs)
   1792 result = None
   1793 called_always_called_hooks = set()

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch_geometric/nn/models/tgn.py:203, in LastAggregator.forward(self, msg, index, t, dim_size)
    202 def forward(self, msg: Tensor, index: Tensor, t: Tensor, dim_size: int):
--> 203     argmax = scatter_argmax(t, index, dim=0, dim_size=dim_size)
    204     out = msg.new_zeros((dim_size, msg.size(-1)))
    205     mask = argmax < msg.size(0)  # Filter items with at least one entry.

File ~/Devs/apple_aiml/c1/tabular/python/.venv/lib/python3.13/site-packages/torch_geometric/utils/_scatter.py:169, in scatter_argmax(src, index, dim, dim_size)
    167 if not is_in_onnx_export():
    168     res = src.new_empty(dim_size)
--> 169     res.scatter_reduce_(0, index, src.detach(), reduce='amax',
    170                         include_self=False)
    171 else:
    172     # `include_self=False` is currently not supported by ONNX:
    173     res = src.new_full(
    174         size=(dim_size, ),
    175         fill_value=src.min(),  # type: ignore
    176     )

RuntimeError: not supported for torch.int64

SyntaxError: unmatched ')' (1289920715.py, line 7)